# 05. Data Quality and Reconciliation

## Banking Transaction Lakehouse

**Purpose:** Verify that records processed through Bronze, Silver and Quarantine are accounted for correctly.

### Scope
- Validate record counts.
- Reconcile Bronze against Silver and Quarantine.
- Persist batch-level data quality results.
- Inspect reconciliation status.
- Demonstrate failure handling.

**Implementation:** Data quality logic is maintained in `src/quality/data_quality.py`.

In [0]:
from pyspark.sql import functions as F

from src.quality.data_quality import reconcile_data_quality

BRONZE_TABLE = "workspace.default.dev_bronze_transactions"
SILVER_TABLE = "workspace.default.demo_silver_transactions"
QUARANTINE_TABLE = "workspace.default.demo_quarantine_transactions"
DQ_TABLE = "workspace.default.dev_etl_dq_summary"

batch_id = "bronze_demo_20261106_01"

print("Data quality notebook initialized.")

In [0]:
# Read records belonging to this batch
bronze_count = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("batch_id") == batch_id)
    .count()
)

valid_count = (
    spark.table(SILVER_TABLE)
    .filter(F.col("batch_id") == batch_id)
    .count()
)

quarantine_batch = (
    spark.table(QUARANTINE_TABLE)
    .filter(F.col("batch_id") == batch_id)
)

quarantine_count = quarantine_batch.count()

duplicate_count = (
    quarantine_batch
    .filter(F.col("dq_reason") == "DUPLICATE_TRANSACTION")
    .count()
)

print("Bronze records:", bronze_count)
print("Valid records:", valid_count)
print("Quarantine records:", quarantine_count)
print("Duplicate records:", duplicate_count)

In [0]:
dq_result = reconcile_data_quality(
    spark=spark,
    batch_id=batch_id,
    bronze_count=bronze_count,
    valid_count=valid_count,
    quarantine_count=quarantine_count,
    duplicate_count=duplicate_count,
    dq_table=DQ_TABLE
)

display(dq_result)

In [0]:
# Read the persisted Data Quality summary
dq_verification = (
    spark.table(DQ_TABLE)
    .filter(F.col("batch_id") == batch_id)
)

display(dq_verification)

In [0]:
# Use a separate batch ID so existing results remain untouched
failure_test_batch = "dq_failure_demo_20261106_01"

try:
    reconcile_data_quality(
        spark=spark,
        batch_id=failure_test_batch,
        bronze_count=bronze_count,
        valid_count=valid_count,
        quarantine_count=quarantine_count,
        duplicate_count=duplicate_count,
        dq_table=DQ_TABLE,
        force_failure=True
    )

except RuntimeError as error:
    print("Expected test failure captured:")
    print(error)